# Simple RecSys Application First

In [11]:
import numpy as np, pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName ("Matrix Work") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

In [ ]:
#print(pair.count())
#print(bridge.count())
#print(bridge.select("pid", "pos").distinct().count())
#print(bridge.filter(F.col("track_uri").isNull() | (F.col("track_uri") == "")).count())   # must be 0

rep = (bridge.groupBy("pid", "track_uri")
       .agg(F.count("*").alias("n"), F.countDistinct("pos").alias("n_pos")).orderBy(F.col("n_pos").desc()))

rep.show()

+------+--------------------+---+-----+
|   pid|           track_uri|  n|n_pos|
+------+--------------------+---+-----+
| 67052|spotify:track:1mr...|138|  138|
|477863|spotify:track:2Wf...| 57|   57|
|218358|spotify:track:2AG...| 56|   56|
|480141|spotify:track:5KY...| 55|   55|
|331576|spotify:track:2rs...| 55|   55|
|652175|spotify:track:7Kc...| 54|   54|
|819513|spotify:track:5fp...| 51|   51|
|481135|spotify:track:4TV...| 50|   50|
|481135|spotify:track:59J...| 50|   50|
|481135|spotify:track:3mA...| 50|   50|
|682157|spotify:track:2HW...| 48|   48|
|437502|spotify:track:3fq...| 47|   47|
|626358|spotify:track:5Bk...| 39|   39|
|873919|spotify:track:3XH...| 33|   33|
|626358|spotify:track:5MI...| 33|   33|
|811998|spotify:track:14W...| 30|   30|
|626358|spotify:track:1Xy...| 30|   30|
|408204|spotify:track:3Ep...| 30|   30|
|461736|spotify:track:2LQ...| 30|   30|
|906273|spotify:track:4DM...| 29|   29|
+------+--------------------+---+-----+
only showing top 20 rows


In [11]:
rep = (bridge.groupBy("pid", "track_uri")
             .agg(F.count("*").alias("n"), F.countDistinct("pos").alias("n_pos"))
             .filter("n > 1"))
rep.agg(F.sum(F.col("n") - 1), F.count("*"), F.max("n"),
        F.sum(F.when(F.col("n_pos") != F.col("n"), 1).otherwise(0))).show()

+------------+--------+------+--------------------------------------------------+
|sum((n - 1))|count(1)|max(n)|sum(CASE WHEN (NOT (n_pos = n)) THEN 1 ELSE 0 END)|
+------------+--------+------+--------------------------------------------------+
|      881652|  829665|   138|                                                 0|
+------------+--------+------+--------------------------------------------------+



In [ ]:
# Collaborative Filtering Approach
# Lets start by generating a matrix of our data. Rows will represent tracks and columns will represent playlists
# For simplicity, we will be using 1s and 0s

# The size of this array is expected to be 2262292 x 1000000 - how to handle array of this size?
# Numpy uses RAM allocation for arrays...meaning trying to do this locally- you'd likely crash 2.26M x 1M  is about 2.26 trillion matrix cells...
# Bring in SciPy

# We can use our bridge table to get distinct pairs, that will help the matrix construction, we'll use the distinct function to be sure track and playlist pairs or unique

bridge = spark.read.parquet("../silver/pid_pos") 
pair = bridge.select("pid", "track_uri").distinct() #we want to turn this into a matrix

# we can use pid itself as the index for playlists 0-999999
# that leaves us with having to index tracks - which should match out track count -> 2262292

w = Window.orderBy("track_uri") #defining window spec

#the following will create a dictionary that assigns an index to the distinct tracks (counts should match our silver count)
track_index = (pair.select("track_uri").distinct()
                   .withColumn("track_idx", F.row_number().over(w) - 1))

track_index.count() #checks out with our track count


2262292

In [ ]:
#now we will start adding to our gold layer

#interactions will serve to map playlist id to the indices we created with track_index
interactions = (pair.join(track_index, on='track_uri', how='inner')
                .select(F.col('pid').cast('int'), F.col('track_idx')))

interactions.count()

65464776


In [10]:
#Lets persist
track_index.coalesce(1).write.mode("overwrite").parquet("../gold/track_index")
interactions.coalesce(8).write.mode("overwrite").parquet("../gold/interactions")

In [14]:
from scipy.sparse import csr_array

N_TRACKS = 2_262_292
N_PLAYLISTs = 1_000_000

df = pd.read_parquet("../gold/interactions")

rows = df["track_idx"].to_numpy()
cols = df["pid"].to_numpy()



In [ ]:
#constructing the matrix
